### 1. Import Modules

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import scipy.sparse as sps
from datetime import datetime
from causality.module.model import SharedNCFPlus, SharedLinearCFPlus
from causality.module.metric import cdcg_func, car_func, cp_func
from causality.module.dataset import load_data, generate_total_sample
from causality.module.utils import set_device, set_seed

### 2. Experiment Settings

In [ ]:
lr = 1e-4
weight_decay = 1e-4
embedding_k = 64
batch_size = 4096
num_epochs = 5
random_seed = 0
evaluate_interval = 50
top_k_list = [10, 30, 100, 1372]
data_dir = "./causality/data"
dataset_name = "personalized"
alpha = 1.
base_model = "ncf"
device = "none"
omega = 9999.
if omega < 9999.:
    omega1 = 1/omega
    omega0 = 1/(1-omega)
else:
    omega1 = 1.
    omega0 = 1.
expt_num = f'{datetime.now().strftime("%y%m%d_%H%M%S_%f")}'
set_seed(random_seed)
device = set_device(device)

### 3. Data Loading

In [ ]:
x_train, x_test = load_data(data_dir, dataset_name)
x_train, y_train, t_train, ps_train = x_train[:,:2].astype(int), x_train[:,2:3], x_train[:,3:4], x_train[:,4:]
x_test, cate_test = x_test[:,:2].astype(int), x_test[:,2]
num_users = x_train[:,0].max()+1
num_items = x_train[:,1].max()+1
print(f"# user: {num_users}, # item: {num_items}")
x_all = generate_total_sample(num_users, num_items)
y1_train = y_train[t_train==1]
x1_train = x_train[np.squeeze(t_train==1)]
ps1_train = ps_train[t_train==1]
obs1 = sps.csr_matrix((np.ones(len(y1_train)), (x1_train[:, 0], x1_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
y1_entire = sps.csr_matrix((y1_train, (x1_train[:, 0], x1_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
ps1_entire = sps.csr_matrix((ps1_train, (x1_train[:, 0], x1_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
y0_train = y_train[t_train==0]
x0_train = x_train[np.squeeze(t_train==0)]
ps0_train = 1-ps_train[t_train==0]
obs0 = sps.csr_matrix((np.ones(len(y0_train)), (x0_train[:, 0], x0_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
y0_entire = sps.csr_matrix((y0_train, (x0_train[:, 0], x0_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
ps0_entire = sps.csr_matrix((ps0_train, (x0_train[:, 0], x0_train[:, 1])), shape=(num_users, num_items), dtype=np.float32).toarray().reshape(-1)
num_samples = len(x_all)
total_batch = num_samples // batch_size
x_test_tensor = torch.LongTensor(x_test).to(device)

### 4. Model Initiallization

In [ ]:
if base_model == "ncf":
    model = SharedNCFPlus(num_users, num_items, embedding_k)
elif base_model == "linearcf":
    model = SharedLinearCFPlus(num_users, num_items, embedding_k)
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)


### 5. Training

In [ ]:
for epoch in range(1, num_epochs+1):
    all_idx = np.arange(num_samples)
    np.random.shuffle(all_idx)
    model.train()
    epoch_total_loss = 0.
    epoch_y1_loss = 0.
    epoch_y0_loss = 0.
    epoch_t_loss = 0.


    for idx in range(total_batch):
        selected_idx = all_idx[batch_size*idx:(idx+1)*batch_size]
        sub_x = x_all[selected_idx]
        sub_x = torch.LongTensor(sub_x).to(device)
        pred_y1, pred_y0, ctr = model(sub_x)


        sub_y = y1_entire[selected_idx]
        sub_y = torch.Tensor(sub_y).unsqueeze(-1).to(device)
        sub_t = obs1[selected_idx]
        sub_t = torch.Tensor(sub_t).unsqueeze(-1).to(device)
        rec_loss = nn.functional.binary_cross_entropy(
            nn.Sigmoid()(pred_y1), sub_y, reduction="none")
        y1_loss = torch.mean(rec_loss * sub_t) * omega1
        ctr_loss = nn.functional.binary_cross_entropy(nn.Sigmoid()(ctr), sub_t) * alpha


        sub_y = y0_entire[selected_idx]
        sub_y = torch.Tensor(sub_y).unsqueeze(-1).to(device)
        sub_t = obs0[selected_idx]
        sub_t = torch.Tensor(sub_t).unsqueeze(-1).to(device)
        rec_loss = nn.functional.binary_cross_entropy(
            nn.Sigmoid()(pred_y0), sub_y, reduction="none")
        y0_loss = torch.mean(rec_loss * sub_t) * omega0


        total_loss = y1_loss + y0_loss + ctr_loss
        epoch_y1_loss += y1_loss
        epoch_t_loss += ctr_loss
        epoch_y0_loss += y0_loss
        epoch_total_loss += total_loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()


    print(f"[Epoch {epoch:>4d} Train Loss] y1: {epoch_y1_loss.item():.4f} / y0: {epoch_y0_loss.item():.4f}")

### 6. Evaluation

In [ ]:
model.eval()
pred_y1, pred_y0, _ = model(x_test_tensor)
pred_y1 = nn.Sigmoid()(pred_y1).detach().cpu().numpy()
pred_y0 = nn.Sigmoid()(pred_y0).detach().cpu().numpy()
pred = (pred_y1 - pred_y0).squeeze()


cdcg_res = cdcg_func(pred, x_test, cate_test, top_k_list)
cdcg_dict: dict = {}
for top_k in top_k_list:
    cdcg_dict[f"cdcg_{top_k}"] = np.mean(cdcg_res[f"cdcg_{top_k}"])
cp_res = cp_func(pred, x_test, cate_test, top_k_list)
cp_dict: dict = {}
for top_k in top_k_list:
    cp_dict[f"cp_{top_k}"] = np.mean(cp_res[f"cp_{top_k}"])
car_res = car_func(pred, x_test, cate_test, top_k_list)
car_dict: dict = {}
for top_k in top_k_list:
    car_dict[f"car_{top_k}"] = np.mean(car_res[f"car_{top_k}"])
mse = np.square(cate_test - pred).mean()


print({"mse":mse})
print(f"cDCG: {cdcg_dict}")
print(f"cP: {cp_dict}")
print(f"cAR: {car_dict}")